In [33]:
import cv2
import numpy as np
import os


# DIRECTORY CONFIG

INPUT_DIR = "input"
OUTPUT_DIR = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# IMAGE LOOP

files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for file_name in files:

    img_path = os.path.join(INPUT_DIR, file_name)
    img = cv2.imread(img_path)

    if img is None:
        print(f"❌ Failed to load {file_name}")
        continue

    result_img = img.copy()

    
    # STEP 1: GRAYSCALE + BLUR
    
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    smooth = cv2.GaussianBlur(gray_img, (7, 7), 0)

    
    # STEP 2: OTSU BINARIZATION
    
    _, binary = cv2.threshold(
        smooth, 0, 255,
        cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU
    )

    
    # STEP 3: CLEANUP USING MORPHOLOGY
    
    morph_kernel = np.ones((5, 5), dtype=np.uint8)

    cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, morph_kernel, iterations=2)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, morph_kernel, iterations=1)

    
    # STEP 4: FIND OBJECTS
    
    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:

        # REMOVE SMALL OBJECTS
        
        area_val = cv2.contourArea(contour)
        if area_val < 6500:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if min(w, h) < 125:
            continue

     
        # SHAPE FEATURES
       
        peri = cv2.arcLength(contour, True)
        if peri == 0:
            continue

        circ = (4 * np.pi * area_val) / (peri * peri)
        ratio = w / float(h)

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)
        if hull_area == 0:
            continue

        solid = area_val / hull_area

        
        # DECISION LOGIC
       
        is_circle = circ > 0.75 and solid > 0.92
        is_square = 0.90 < ratio < 1.25 and solid > 0.95
        
        if is_circle or is_square:
            tag = "Intact Biscuit"
            box_color = (0, 255, 0)
        else:
            tag = "Broken Biscuit"
            box_color = (0, 0, 255)

        
        # DRAW OUTPUT
        
        cv2.rectangle(result_img, (x, y), (x + w, y + h), box_color, 2)

        cv2.putText(result_img, tag, (x, y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, box_color, 2)

        info = f"C:{circ:.2f} S:{solid:.2f}"
        cv2.putText(result_img, info, (x, y + h + 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

  
    # SAVE RESULT
   
    output_path = os.path.join(OUTPUT_DIR, f"result_{file_name}")
    cv2.imwrite(output_path, result_img)

    print(f"✅ Output saved: {output_path}")

print("🎉 All images processed successfully!")

✅ Output saved: output\result_B-1.jpeg
✅ Output saved: output\result_B-10.jpeg
✅ Output saved: output\result_B-2.jpeg
✅ Output saved: output\result_B-3.jpeg
✅ Output saved: output\result_B-4.jpeg
✅ Output saved: output\result_B-5.jpeg
✅ Output saved: output\result_B-6.jpeg
✅ Output saved: output\result_B-7.jpeg
✅ Output saved: output\result_B-8.jpeg
✅ Output saved: output\result_B-9.jpg
🎉 All images processed successfully!
